A task that has proven somewhat tricky is to obtain the optimal gate for a given angle automatically. 

What we can do is to use a neural network with just one layer as the time NN. This is terribly inadequate for the multi-angle optimization but should do just right for single-angle optimization to find the minimum time solution. 

In [ ]:
import torch 
from decimal import Decimal
from schsolve import neural_trainer, schsolver 
import torchdiffeq as tdf 
import cst_n_fn as cfn 
import matplotlib.pyplot as plt 
import const_czphi as czphi 
import numpy as np 
from const_czphi import reduce_r_dim_2q_vector, correction_1q

device = (
    "cuda"
    if torch.cuda.is_available()
    else "cpu"
)

device = "cpu"

print(device)
    
# **** training function below ****
# Note that this has to be coded separately for each use case 
options_dict = {'dtype':torch.float64}

def trainer(neural_model, nn_solver_1, nn_solver_2, input_tensor, epoch, init_matrix, multiplier = 2.0, print_ = True):
    
    neural_model.train()
    nn_solver_1.zero_grad()
    nn_solver_2.zero_grad()
    
    nn_time_output = (neural_model(input_tensor).select(1,0))
    pred_outputs_detuning = nn_time_output*(czphi.range_detuning[1] - czphi.range_detuning[0]) + czphi.range_detuning[0]
        
    czphi.instance.hamiltonian.rabi_tensored["pulse 0"] \
    = czphi.instance.hamiltonian.rabi_tensored["pulse 1"] \
    = cfn.const_then_zero_tensor(cfn.rabi, neural_model.gatetime_prediction)

    czphi.instance.hamiltonian.det_tensored["pulse 0"] \
    = czphi.instance.hamiltonian.det_tensored["pulse 1"] \
    = czphi.list_to_fn_tensor_var_gatetime(pred_outputs_detuning, neural_model.gatetime_prediction, czphi.time_steps)    
    
    time_arr_ = torch.linspace(0, 1.0, 3, device = device)*neural_model.gatetime_prediction.max()#neural_model.gatetime_prediction.max()#neural_model.gatetime_prediction.max()
    
    # * code below evolves the Hamiltonian with time-dependent controls and computes the state in the subspace of 0s and 1s

    sol_intrm = reduce_r_dim_2q_vector(tdf.odeint(czphi.instance,\
                                init_matrix, time_arr_, 
                                method = 'dopri5',rtol=1e-5, atol=1e-5, \
                                options = options_dict)[-1], angle_batch = czphi.angle_batch)
    
    solution = correction_1q(sol_intrm, angle_batch = czphi.angle_batch) 

    infidelity_term = cfn.unitary_infidelity_array(solution, cfn.czp_gate_stack(input_tensor), nqbits = 2)  
    time_term = multiplier*torch.mean(neural_model.gatetime_prediction)

    # reg_term = 3e0*(torch.abs(7.613/cfn.rabi - neural_model.gatetime_prediction.max()))

    dict_ = {"time term": time_term.item(), "infidelity term": infidelity_term.item(),
                "gatetime mean.": torch.mean(neural_model.gatetime_prediction)}#, 'reg_term': reg_term}
    # ! save below 
    loss_instance = infidelity_term + time_term #+ reg_term
    
    neural_model.debug_gradient_time.retain_grad()
    loss_instance.backward()
    nn_solver_2.step()
    nn_solver_1.step()
    
    if epoch%25 == 0 and print_ == True:  
        print(neural_model.gatetime_prediction.max()*cfn.rabi)
        
        print("Epoch {}: Loss = {:.2E}".format(epoch, Decimal(loss_instance.cpu().detach().numpy().item())))
        
        print(dict_)
        #        print(neural_model.debug_gradient_time.grad)
    #if epoch%2e2 == 0:
    #    print(nn_time_output)
    
    return solution.cpu().detach().numpy(), loss_instance.item(), dict_
